In [1]:
!apt-get install -y libpango1.0-dev libcairo2-dev pkg-config python3-dev
!pip install manim numpy
!apt-get install -y texlive texlive-latex-extra texlive-fonts-extra dvipng

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pkg-config is already the newest version (0.29.2-1ubuntu3).
libcairo2-dev is already the newest version (1.16.0-5ubuntu2.1).
libpango1.0-dev is already the newest version (1.50.6+ds-2ubuntu1).
python3-dev is already the newest version (3.10.6-1~22.04.1).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  dvisvgm fonts-adf-accanthis fonts-adf-berenis fonts-adf-gillius
  fonts-adf-universalis fonts-cabin fonts-cantarell f

In [6]:
%%manim -ql CNNLayerVisualizer

from manim import *
import numpy as np

class CNNLayerVisualizer(Scene):
    def construct(self):
        self.camera.background_color = WHITE
        Text.set_default(color=BLACK)

        # Sequence of Animations
        self.show_convolution()
        self.show_relu()
        self.show_max_pooling()

    def show_convolution(self):
        """Visualizes tf.nn.conv2d + tf.nn.bias_add"""
        title = Text("1. Convolution (Apply Kernel)", font_size=40).to_edge(UP)
        self.play(Write(title))

        # Create an input "Image" Matrix (5x5)
        input_matrix = IntegerMatrix(
            np.random.randint(0, 5, (5, 5)),
            v_buff=0.8, h_buff=0.8
        ).set_color(BLACK).shift(LEFT * 3)
        input_label = Text("Input Image", font_size=24).next_to(input_matrix, DOWN)

        # Create an output Matrix (3x3 for a 3x3 kernel with no padding)
        output_matrix = IntegerMatrix(
            np.zeros((3, 3), dtype=int),
            v_buff=0.8, h_buff=0.8
        ).set_color(BLACK).shift(RIGHT * 3)
        output_label = Text("Feature Map", font_size=24).next_to(output_matrix, DOWN)

        self.play(FadeIn(input_matrix), FadeIn(input_label), FadeIn(output_matrix), FadeIn(output_label))

        # Sliding Window (Kernel representation)
        # Highlight a 3x3 section of the input matrix
        kernel_box = SurroundingRectangle(
            VGroup(*[input_matrix.get_entries()[i] for i in [0, 1, 2, 5, 6, 7, 10, 11, 12]]),
            color=BLUE, stroke_width=4
        )
        kernel_label = Text("Kernel (3x3)", font_size=20, color=BLUE).next_to(kernel_box, UP)

        self.play(Create(kernel_box), Write(kernel_label))

        # High level conceptual animation: sliding the kernel
        # We slide the box across the top row to show the 'stride'
        for i in range(3):
            # Target a single output pixel to show the calculation mapping
            target_pixel = output_matrix.get_entries()[i]
            highlight_pixel = SurroundingRectangle(target_pixel, color=RED, stroke_width=4)

            self.play(
                Create(highlight_pixel),
                Indicate(target_pixel, color=RED),
                run_time=0.5
            )
            self.play(FadeOut(highlight_pixel), run_time=0.2)

            # Move kernel to the right (stride = 1)
            if i < 2:
                self.play(
                    kernel_box.animate.shift(RIGHT * 0.8), # Shift matches h_buff
                    kernel_label.animate.shift(RIGHT * 0.8)
                )

        self.wait(1)
        self.play(FadeOut(Group(input_matrix, input_label, output_matrix, output_label, kernel_box, kernel_label, title)))

    def show_relu(self):
        """Visualizes tf.nn.relu"""
        title = Text("2. ReLU Activation", font_size=40).to_edge(UP)
        self.play(Write(title))

        # Matrix containing negative values
        matrix_data = [[-2, 5, -1], [3, -4, 0], [7, -8, 2]]
        relu_matrix = IntegerMatrix(matrix_data, v_buff=0.8, h_buff=0.8).set_color(BLACK)

        self.play(FadeIn(relu_matrix))

        # Animate ReLU logic: max(0, x)
        # Change all negative numbers to 0
        transformations = []
        for i, row in enumerate(matrix_data):
            for j, val in enumerate(row):
                if val < 0:
                    entry = relu_matrix.get_entries()[i * 3 + j]
                    # Create a new '0' to replace the negative number
                    zero_text = Integer(0).move_to(entry).set_color(GREEN)
                    transformations.append(Transform(entry, zero_text))

        self.play(*transformations, run_time=2)

        relu_label = Text("Negative values become 0", font_size=24, color=GREEN).next_to(relu_matrix, DOWN)
        self.play(FadeIn(relu_label))

        self.wait(1)
        self.play(FadeOut(Group(relu_matrix, relu_label, title)))

    def show_max_pooling(self):
        """Visualizes tf.nn.max_pool2d"""
        title = Text("3. Max Pooling (2x2, Stride 2)", font_size=40).to_edge(UP)
        self.play(Write(title))

        # 4x4 Input Matrix
        pool_input = IntegerMatrix(
            [[1, 3, 2, 4], [5, 6, 1, 2], [0, 1, 8, 3], [2, 1, 4, 7]],
            v_buff=0.8, h_buff=0.8
        ).set_color(BLACK).shift(LEFT * 3)

        # 2x2 Output Matrix
        pool_output = IntegerMatrix(
            [[0, 0], [0, 0]],
            v_buff=1.0, h_buff=1.0
        ).set_color(BLACK).shift(RIGHT * 3)

        self.play(FadeIn(pool_input), FadeIn(pool_output))

        # Highlight Top-Left 2x2 block
        top_left_indices = [0, 1, 4, 5]
        block = VGroup(*[pool_input.get_entries()[i] for i in top_left_indices])
        pool_box = SurroundingRectangle(block, color=ORANGE, stroke_width=4)

        self.play(Create(pool_box))

        # The maximum value in that block is 6 (at index 5 in a flattened list)
        max_val_entry = pool_input.get_entries()[5]
        target_entry = pool_output.get_entries()[0]

        # Animate the maximum value jumping to the output
        self.play(Indicate(max_val_entry, color=ORANGE, scale_factor=1.5))

        # Transform output '0' to '6'
        new_val = Integer(6).move_to(target_entry).set_color(ORANGE)
        self.play(Transform(target_entry, new_val))

        # Slide to the next 2x2 block (Stride 2)
        self.play(pool_box.animate.shift(RIGHT * 1.6)) # Shift 2 columns

        self.wait(2)
        self.play(FadeOut(Group(pool_input, pool_output, pool_box, title)))

Manim Community v0.20.1

[05/14/26 16:36:33] INFO     Animation 0 : Partial movie file written in                   ]8;id=668161;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=6679;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/CNN                         
                             LayerVisualizer/793938918_1038270762_223132457.mp4'                                   

                    INFO     Writing \special{dvisvgm:raw <g                                ]8;id=302055;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=970847;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#110\110]8;;\
                             id='unique000'>}4\special{dvisvgm:raw </g>} to                                        
                             media/Tex/b6e1e0d35a22cf8a.tex                                                        

[05/14/26 16:36:34] INFO     Writing \special{dvisvgm:raw <g                                ]8;id=656396;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=112847;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#110\110]8;;\
                             id='unique000'>}2\special{dvisvgm:raw </g>} to                                        
                             media/Tex/92c26d21f4d400ae.tex                                                        

[05/14/26 16:36:35] INFO     Writing \special{dvisvgm:raw <g                                ]8;id=302235;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=64758;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#110\110]8;;\
                             id='unique000'>}3\special{dvisvgm:raw </g>} to                                        
                             media/Tex/9ca767606662f643.tex                                                        

...

[05/14/26 16:37:13] INFO     Animation 25 : Partial movie file written in                  ]8;id=157440;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=319770;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/CNN                         
                             LayerVisualizer/2791924351_2230675760_1381065772.mp4'                                 

[05/14/26 16:37:15] INFO     Animation 26 : Partial movie file written in                  ]8;id=712296;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=332487;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/CNN                         
                             LayerVisualizer/2791924351_771346772_2420296903.mp4'                                  

                    INFO     Combining to Movie file.                                      ]8;id=404993;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=866208;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=88666;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=890461;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/content/media/videos/content/480p15/CNNLayerVisualizer.mp4'                         
                                                                                                                   

                    INFO     Rendered CNNLayerVisualizer                                               ]8;id=315678;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=555629;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py#278\278]8;;\
                             Played 27 animations                                                                  

In [7]:
import subprocess
import subprocess
result = subprocess.run(['find', '/', '-name', '*.mp4', '-not', '-path', '*/proc/*'], capture_output=True, text=True)
print(result.stdout)

/usr/local/lib/python3.12/dist-packages/panel/tests/pane/assets/mp4.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/a.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/b.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/world.mp4
/usr/share/texlive/texmf-dist/tex/latex/mwe/example-movie.mp4
/content/media/videos/content/480p15/partial_movie_files/CNNLayerVisualizer/2791924351_882277631_2873729818.mp4
/content/media/videos/content/480p15/partial_movie_files/CNNLayerVisualizer/793938918_1038270762_223132457.mp4
/content/media/videos/content/480p15/partial_movie_files/CNNLayerVisualizer/2791924351_1039292708_3399569858.mp4
/content/media/videos/content/480p15/partial_movie_files/CNNLayerVisualizer/2791924351_2505866458_3867285389.mp4
/content/media/videos/content/480p15/partial_movie_files/CNNLayerVisualizer/2791924351_3908649253_2430205266.mp4
/content/media/videos/content/480p15/partial_movie_files/CNNLayerVisualizer/27919

In [8]:
from google.colab import files

files.download('/content/media/videos/content/480p15/CNNLayerVisualizer.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>